# SFT LoRA with Cut Tokenizer (Qwen3-4B-Thinking)

This notebook mirrors NVARC's tokenizer cutting approach, then runs LoRA fine-tuning and evaluation on the cut model.

## Configuration

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Config
from dataclasses import dataclass
import os
import random
import torch

@dataclass
class Config:
    model_path: str = 'models/Qwen3-4B-Thinking-2507'
    cut_output_dir: str = 'outputs/qwen3-4b-thinking-16tok-cut'
    max_vocab_size: int = 16

    data_root: str = 'data'
    train_split: str = 'training'
    eval_split: str = 'evaluation'
    max_train_tasks: int = 0
    max_eval_tasks: int = 0
    use_augmentation: bool = False
    aug_samples_per_task: int = 1
    aug_seed: int = 0

    max_seq_len: int = 1024
    max_new_tokens: int = 512

    lora_r: int = 64
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )

    output_dir: str = 'outputs/arc_lora_sft_cut'
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-4
    num_train_epochs: int = 3
    logging_steps: int = 10
    save_steps: int = 200

cfg = Config()

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Cut tokenizer vocab to fixed 16-token set (no chat_template needed)
from transformers import AutoTokenizer
import json

tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)

# check if cut vocab is in original tokenizer
required = ['<|im_start|>', '<|im_end|>', '<|endoftext|>', '\n', 'user', 'assistant'] + [str(i) for i in range(10)]
vocab = tokenizer.get_vocab()
missing = [t for t in required if t not in vocab]
if missing:
    print('missing tokens in vocab:', missing)
    raise ValueError(f'missing tokens in vocab: {missing}')

# keep exactly these 16 tokens
mapping = [vocab[tok] for tok in required]
new_vocab = {tok: idx for idx, tok in enumerate(required)}

print('vocab size:', len(new_vocab))
print(new_vocab)

os.makedirs(cfg.cut_output_dir, exist_ok=True)
with open(os.path.join(cfg.cut_output_dir, 'mapping.json'), 'w', encoding='utf-8') as f:
    json.dump({'old_ids': mapping, 'tokens': required}, f, ensure_ascii=False, indent=2)

In [ ]:
# Slice embedding/lm_head weights across safetensors shards
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoConfig
import glob

shards = sorted(glob.glob(os.path.join(cfg.model_path, '*.safetensors')))
assert shards, 'No safetensors found in model_path'

row_select = torch.tensor(mapping)

for shard in shards:
    tensors = {}
    with safe_open(shard, framework='pt', device='cpu') as f:
        for key in f.keys():
            tensors[key] = f.get_tensor(key)

    if 'model.embed_tokens.weight' in tensors:
        tensors['model.embed_tokens.weight'] = torch.index_select(
            tensors['model.embed_tokens.weight'], 0, row_select
        )
    if 'lm_head.weight' in tensors:
        tensors['lm_head.weight'] = torch.index_select(
            tensors['lm_head.weight'], 0, row_select
        )

    out_path = os.path.join(cfg.cut_output_dir, os.path.basename(shard))
    save_file(tensors, out_path)

# update config vocab_size
config = AutoConfig.from_pretrained(cfg.model_path)
config.vocab_size = len(new_vocab)
config.save_pretrained(cfg.cut_output_dir)

# save minimal tokenizer files
tokenizer.save_pretrained(cfg.cut_output_dir)
with open(os.path.join(cfg.cut_output_dir, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(new_vocab, f, ensure_ascii=False, indent=2)

## Preparation

In [ ]:
# Build SFT training samples
from eval.core import GridCodec, ARCDataset
from ARChitects.data import AugmentedARCDataset
from ARChitects.sft_utils import build_sft_samples

codec = GridCodec()
base_train = ARCDataset(root=cfg.data_root, split=cfg.train_split, max_tasks=cfg.max_train_tasks)
if cfg.use_augmentation:
    train_source = AugmentedARCDataset(
        base=base_train,
        samples_per_task=cfg.aug_samples_per_task,
        seed=cfg.aug_seed,
    )
else:
    train_source = base_train

samples = build_sft_samples(train_source, codec)
print('samples:', len(samples))

In [ ]:
# Load cut model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(cfg.cut_output_dir, use_fast=False)
if tokenizer.eos_token_id is None:
    eos_id = tokenizer.convert_tokens_to_ids('<|endoftext|>')
    if eos_id is not None and eos_id != tokenizer.unk_token_id:
        tokenizer.eos_token_id = eos_id

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else '<|endoftext|>'

model = AutoModelForCausalLM.from_pretrained(
    cfg.cut_output_dir,
    dtype=getattr(torch, 'bfloat16', None),
    device_map='auto',
)
model.config.use_cache = False

## Baseline

In [ ]:
# Baseline evaluation with cut model (before LoRA)
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core import run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

eval_dataset = ARCDataset(root=cfg.data_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
baseline_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
baseline_solver = RawSolver(model=baseline_model, codec=codec)
baseline_reports = run_evaluation(
    dataset=eval_dataset,
    solver=baseline_solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports_baseline'),
    model_id=cfg.cut_output_dir,
    model_key='baseline_cut',
    viz_failures=True,
)
print(baseline_reports['summary'])

## LoRA

In [ ]:
# Freeze base; train LoRA only
from peft import LoraConfig, get_peft_model

for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Build PyTorch Dataset and collator
from ARChitects.sft_utils import ArcSFTDataset, collate_sft

train_dataset = ArcSFTDataset(samples, tokenizer, max_seq_len=cfg.max_seq_len)
def collate_fn(batch):
    return collate_sft(batch, tokenizer=tokenizer)

In [ ]:
# Train LoRA
from transformers import TrainingArguments, Trainer

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    bf16=bf16_ok,
    fp16=torch.cuda.is_available() and not bf16_ok,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

trainer.train()

In [ ]:
# Save LoRA adapter
adapter_dir = os.path.join(cfg.output_dir, 'lora_adapter')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('saved:', adapter_dir)

## Evalution

In [ ]:
# Evaluate with eval/ and write reports
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core import run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

eval_dataset = ARCDataset(root=cfg.data_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
eval_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
solver = RawSolver(model=eval_model, codec=codec)

reports = run_evaluation(
    dataset=eval_dataset,
    solver=solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports'),
    model_id=cfg.cut_output_dir,
    model_key='lora_sft_cut',
    viz_failures=True,
)

print(reports['summary'])

In [ ]:
# Save Lora-ed model as a merged one
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(cfg.cut_output_dir, device_map='auto')
lora = PeftModel.from_pretrained(base, os.path.join(cfg.output_dir, 'lora_adapter'))
merged = lora.merge_and_unload()
merged.save_pretrained(os.path.join(cfg.output_dir, 'merged'))
tokenizer.save_pretrained(os.path.join(cfg.output_dir, 'merged'))